# Challenge 02 — Build a News Agent (Code-First)

In this challenge, you'll build a **News Agent** programmatically using the **Azure AI Agent SDK** and Python. By the end, you'll be able to chat with your agent from this notebook.

## Learning Objectives
- Authenticate to Azure AI Foundry from code (using `DefaultAzureCredential` — no API keys)
- Create an agent with custom system instructions
- Manage the conversation lifecycle: thread → message → run → response
- Inspect and iterate on agent responses

## How This Notebook Works
- **Pre-filled cells** (✅): Environment setup, authentication, and validation — these are done for you.
- **Fill-in-the-blank cells** (🔨): Agent creation, messaging, and run processing — you complete the `___` placeholders.
- Each `___` has a `# TODO:` comment explaining what goes there.

---
## ✅ Setup — Install Dependencies

Run this cell once to install the required packages into your active environment.

In [ ]:
# Install the core packages for this challenge
# This cell is complete — just run it.
%pip install azure-ai-projects azure-ai-agents azure-identity python-dotenv ipykernel --quiet

---
## ✅ Imports & Environment Variables

This cell loads your `.env` file and reads the values you configured in Challenge 00. **Nothing to change here** — just run it and confirm the output shows your endpoint and model.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import MessageRole

# Load environment variables from your .env file
load_dotenv()

# Read values from .env — these map to what you deployed in Challenge 00
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")  # optional but recommended

# Validate that required values are present
assert PROJECT_ENDPOINT, "PROJECT_ENDPOINT is missing from .env"
assert MODEL_DEPLOYMENT_NAME, "MODEL_DEPLOYMENT_NAME is missing from .env"
assert "/api/projects/" in PROJECT_ENDPOINT, (
    "PROJECT_ENDPOINT must be a Foundry *project* endpoint, e.g. "
    "https://<resource>.services.ai.azure.com/api/projects/<project-name> "
    f"(got: {PROJECT_ENDPOINT})"
)

print(f"✅ Endpoint: {PROJECT_ENDPOINT}")
print(f"✅ Model: {MODEL_DEPLOYMENT_NAME}")
print(f"✅ Tenant: {TENANT_ID or '(not set — will use CLI default)'}")

---
## ✅ Authenticate & Create the AgentsClient

This cell authenticates using `ChainedTokenCredential` (tries Azure CLI first, falls back to browser login) and creates the `AgentsClient` you'll use for the rest of the notebook. **Nothing to change here.**

In [ ]:
# Authenticate — no API keys, just Azure Identity
credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

# Create the AgentsClient connected to your Foundry project
agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
print("✅ Connected to Azure AI Foundry")

---
## 🔨 Part 1: Create the News Agent

Now it's your turn. Create an agent named `NewsAgent` with system instructions that define its persona as a travel news briefing assistant.

**Fill in the blanks below.** Each `___` has a hint in the comment above it.

In [ ]:
# TODO: Create your NewsAgent by filling in the blanks
#
# Requirements:
#   - model: Use the model deployment name from your .env
#   - name: "NewsAgent"
#   - instructions: Define a persona that:
#       * Summarizes news topics in a concise, neutral, informative tone
#       * Organizes briefings with clear headlines and short summaries
#       * Covers multiple perspectives when discussing controversial topics
#       * Clarifies it does NOT have live news — uses training knowledge

news_agent = agents_client.create_agent(
    model=___,  # TODO: What variable holds your model deployment name?
    name="___",  # TODO: What should this agent be called?
    instructions=(
        "___"  # TODO: Write the system instructions for the agent
        # Hint: Be specific about tone, format, and limitations.
        # The more detail you give, the better the agent behaves.
    ),
)
print(f"✅ Agent created — ID: {news_agent.id}")
print(f"   Name: {news_agent.name}")
print(f"   Model: {news_agent.model}")

---
## 🔨 Part 2: Create a Conversation Thread

Agents communicate through **threads**. A thread is a conversation session that holds the message history. You create a thread, then add messages to it.

**Fill in the blank** to create a new thread.

In [ ]:
# TODO: Create a conversation thread
#
# Hint: Which method on agents_client creates a new thread?

thread = agents_client.___  # TODO: Call the method that creates a thread
print(f"✅ Thread created — ID: {thread.id}")

---
## 🔨 Part 3: Send a Message to the Agent

Add a user message to the thread. The agent won't respond yet — that happens when you create a **run** in the next step.

**Fill in the blanks** for the message parameters.

In [ ]:
# TODO: Send a user message to the thread
#
# Parameters needed:
#   - thread_id: the ID of the thread you just created
#   - role: this is a USER message (Hint: MessageRole.___)
#   - content: the text of the message

user_message = "What's happening in Texas this week that a traveler should know?"

message = agents_client.messages.create(
    thread_id=___,  # TODO: Which thread does this message belong to?
    role=___,  # TODO: What role is the sender? (Hint: MessageRole has USER and AGENT)
    content=___,  # TODO: What content are you sending?
)
print(f"✅ Message sent: '{user_message}'")

---
## 🔨 Part 4: Run the Agent and Get Its Response

A **run** triggers the agent to process all unread messages in the thread and generate a response. You need to:
1. Create a run (specifying which thread and which agent)
2. Wait for the run to complete
3. Read the messages back from the thread

**Fill in the blanks** to execute the run and read the response.

In [ ]:
# TODO: Create a run and wait for the agent to respond
#
# Hint: agents_client.runs.create_and_process() creates a run AND waits
# for it to finish — it's a convenience method so you don't need to poll.
#
# Parameters needed:
#   - thread_id: which thread to run the agent on
#   - agent_id: which agent should process the messages

run = agents_client.runs.create_and_process(
    thread_id=___,  # TODO: Which thread?
    agent_id=___,  # TODO: Which agent? (Hint: what property of news_agent holds its ID?)
)
print(f"✅ Run complete — Status: {run.status}")

In [ ]:
# TODO: List and print the messages from the thread
#
# After a run completes, the agent's response is added to the thread.
# List all messages and print them to see the full conversation.
#
# Hint: agents_client.messages.list() takes a thread_id
# Each message has: .role (USER or AGENT) and .content[0].text.value

messages = agents_client.messages.list(thread_id=___)  # TODO: Which thread?

for msg in messages:
    role = msg.role
    text = ___  # TODO: How do you get the text from msg.content?
    # Hint: msg.content is a list; the first item has a .text.value property
    print(f"\n--- {role.upper()} ---\n{text}")

---
## 🔨 Part 5: Send a Second Message

The thread preserves conversation history. Send a follow-up message and run the agent again to see multi-turn behavior.

**Fill in the blanks** — same pattern as Part 3 and Part 4.

In [ ]:
# TODO: Send a second message and run the agent again
#
# This tests multi-turn conversation — the agent should remember
# the context from the first message.

second_message = "What are the key things happening in AI regulation right now?"

# Send the message
agents_client.messages.create(
    thread_id=___,  # TODO
    role=___,  # TODO
    content=___,  # TODO
)
print(f"✅ Second message sent: '{second_message}'")

# Run the agent again on the same thread
run2 = agents_client.runs.create_and_process(
    thread_id=___,  # TODO
    agent_id=___,  # TODO
)
print(f"✅ Run complete — Status: {run2.status}")

# Print all messages (full conversation history)
messages2 = agents_client.messages.list(thread_id=___) # TODO
for msg in messages2:
    role = msg.role
    text = msg.content[0].text.value if msg.content else ""
    print(f"\n{'='*50}")
    print(f"{role.upper()}:")
    print(text)

---
## 🔨 Part 6: Verify Your Agent in the Portal

Your agent now exists in your Foundry project. Before moving on:
1. Open the [Microsoft Foundry portal](https://ai.azure.com)
2. Navigate to your project
3. Find your `NewsAgent` in the agents list
4. Confirm it shows the model and instructions you configured

Run the cell below to print the details you should verify in the portal.

In [ ]:
# This cell is complete — just run it to see your agent details
print("=" * 50)
print("YOUR AGENT DETAILS (verify these in the portal):")
print("=" * 50)
print(f"  Agent ID:      {news_agent.id}")
print(f"  Agent Name:    {news_agent.name}")
print(f"  Model:         {news_agent.model}")
print(f"  Instructions:  {news_agent.instructions[:100]}...")
print(f"\n  Thread ID:     {thread.id}")
print(f"  Last Run:      {run.status}")
print("\n⚠️  DO NOT delete this agent — you need it for Challenge 03!")

---
## ⚠️ Cleanup (Only If Starting Over)

**Do NOT run this cell** unless you need to start over. Your `NewsAgent` is needed for Challenge 03 (Tools) and beyond.

In [ ]:
# ONLY run this if you need to start fresh — uncomment the lines below:

# agents_client.delete_agent(news_agent.id)
# print(f"Agent deleted — ID: {news_agent.id}")

# agents_client.threads.delete(thread.id)
# print(f"Thread deleted — ID: {thread.id}")

print("ℹ️ Agent preserved for Challenge 03.")